In [2]:
%load_ext autoreload
%autoreload 2

In [ ]:
import jax
import jax.numpy as jnp

from deltanet.recurrent import compute_recurrent_state
from deltanet.utils import new_key, normalize

steps = [
    1,
    50,
    100,
    1000,
    10_000,
    100_000,
]

T = max(steps)
d_key = 8
d_query = 8
d_value = 8

key_k, key_v, key_q, key_betas = new_key(split=4)

keys = jax.jit(normalize)(jax.random.normal(key_k, (T, d_key)))
queries = jax.jit(normalize)(jax.random.normal(key_q, (T, d_query)))
values = jax.jit(normalize)(jax.random.normal(key_v, (T, d_value)))
betas = jax.jit(normalize)(jnp.ones((T, 1)))

k0, v0 = normalize(keys[0, :]), normalize(values[0, :])

for s in steps:
    updated_state, _ = compute_recurrent_state(
        keys[:s, :], values[:s, :], queries[:s, :], betas[:s]
    )
    o = k0 @ updated_state
    err = jnp.sum(jnp.square(o - v0))
    print(f"---steps={s}, retrieval error: {err}---")

---steps=1, retrieval error: 0.9999993443489075---
---steps=50, retrieval error: 0.9999991059303284---
---steps=100, retrieval error: 0.9999990463256836---
---steps=1000, retrieval error: 0.9999995231628418---
---steps=10000, retrieval error: 1.0000085830688477---
---steps=100000, retrieval error: 1.0000114440917969---
